In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from datetime import datetime

def load_idx_images(filepath):
    import struct
    with open(filepath, 'rb') as f:
        magic, num, rows, cols = struct.unpack('>IIII', f.read(16))
        images = np.frombuffer(f.read(), dtype=np.uint8).reshape(num, rows, cols)
    return images

def load_idx_labels(filepath):
    import struct
    with open(filepath, 'rb') as f:
        magic, num = struct.unpack('>II', f.read(8))
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return labels

def prepare_data(images, labels, num_classes=62):
    X = images.astype('float32') / 255.0
    X = np.expand_dims(X, axis=-1)  
    
    y = keras.utils.to_categorical(labels, num_classes)
    
    return X, y

def create_model(num_classes=62, input_shape=(28, 28, 1)):
    model = keras.Sequential([
        # Conv Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Conv Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Conv Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Fully Connected
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

def get_callbacks(model_name, initial_lr=0.001):
    checkpoint = keras.callbacks.ModelCheckpoint(
        f'{model_name}_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
    
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
    
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    )
    
    def lr_schedule(epoch):
        if epoch < 15:
            return initial_lr
        elif epoch < 30:
            return initial_lr * 0.5
        elif epoch < 45:
            return initial_lr * 0.1
        else:
            return initial_lr * 0.01
    
    lr_scheduler = keras.callbacks.LearningRateScheduler(lr_schedule, verbose=1)
    
    return [checkpoint, early_stop, reduce_lr, lr_scheduler]

def plot_history(history, dataset_name):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    axes[0].plot(history.history['accuracy'], label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[0].set_title(f'{dataset_name} - Accuracy', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['loss'], label='Train Loss')
    axes[1].plot(history.history['val_loss'], label='Val Loss')
    axes[1].set_title(f'{dataset_name} - Loss', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{dataset_name}_training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

def evaluate_model(model, X_test, y_test, dataset_name):
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    
    print(f"\n{'='*60}")
    print(f"{dataset_name} - FINAL RESULTS")
    print(f"{'='*60}")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
    print(f"{'='*60}\n")
    
    with open('training_results.txt', 'a') as f:
        f.write(f"\n{'='*60}\n")
        f.write(f"{dataset_name} - Results\n")
        f.write(f"{'='*60}\n")
        f.write(f"Test Loss: {test_loss:.4f}\n")
        f.write(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)\n")
        f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    return test_loss, test_acc

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

upload_dir = "/content/drive/MyDrive/data"

os.makedirs(upload_dir, exist_ok=True)

print("Upload directory ready:", upload_dir)

In [ ]:
from google.colab import drive
import zipfile, os, shutil

drive.mount('/content/drive')

zips = {
    "original":  "/content/drive/MyDrive/data/original.zip",
    "noisy":     "/content/drive/MyDrive/data/noisy.zip",
    "combined":  "/content/drive/MyDrive/data/combined.zip",
    "balanced":  "/content/drive/MyDrive/data/balanced.zip",
}

base_dir = "/content/data"
os.makedirs(base_dir, exist_ok=True)

for name, zip_path in zips.items():
    print(f"\nExtracting {name}.zip")

    tmp = f"/content/tmp"
    out = f"{base_dir}/{name}"

    if os.path.exists(tmp):
        shutil.rmtree(tmp)
    os.makedirs(tmp)

    with zipfile.ZipFile(zip_path) as z:
        z.extractall(tmp)

    files = os.listdir(tmp)
    src = os.path.join(tmp, files[0]) if len(files) == 1 else tmp

    if os.path.exists(out):
        shutil.rmtree(out)
    shutil.move(src, out)

    shutil.rmtree(tmp)
    print(f"Done → {out}")

In [ ]:
print("TRAINING ON ORIGINAL DATASET")

# Load data
def rotate(images):
    images = np.transpose(images, (0, 2, 1))
    images = np.flip(images, axis=2)
    return images

train_images = load_idx_images('../data/original/emnist-byclass-train-images-idx3-ubyte')
train_labels = load_idx_labels('../data/original/emnist-byclass-train-labels-idx1-ubyte')
test_images = load_idx_images('../data/original/emnist-byclass-test-images-idx3-ubyte')
test_labels = load_idx_labels('../data/original/emnist-byclass-test-labels-idx1-ubyte')

train_images = rotate(train_images)
test_images = rotate(test_images)

print(f"Train: {len(train_images)} images")
print(f"Test: {len(test_images)} images\n")

# Prepare data
X_train, y_train = prepare_data(train_images, train_labels, num_classes=62)
X_test, y_test   = prepare_data(test_images, test_labels, num_classes=62)

# Create and compile model
model_original = create_model()
model_original.summary()

initial_lr = 0.001
optimizer = keras.optimizers.Adam(learning_rate=initial_lr)
model_original.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history_original = model_original.fit(
    X_train, y_train,
    batch_size=128,
    epochs=60,
    validation_split=0.1,
    callbacks=get_callbacks('emnist_original_model'),
    verbose=1
)

# Plot history
plot_history(history_original, 'Original Dataset')

# Evaluate
test_loss_orig, test_acc_orig = evaluate_model(model_original, X_test, y_test, 'ORIGINAL DATASET')

model_original.save('models/emnist_original.keras')

In [ ]:
print("TRAINING ON BALANCED DATASET")

# Load data
train_images = load_idx_images('../data/balanced/emnist-byclass-train-images-idx3-ubyte')
train_labels = load_idx_labels('../data/balanced/emnist-byclass-train-labels-idx1-ubyte')
test_images = load_idx_images('../data/balanced/emnist-byclass-test-images-idx3-ubyte')
test_labels = load_idx_labels('../data/balanced/emnist-byclass-test-labels-idx1-ubyte')

print(f"Train: {len(train_images)} images")
print(f"Test: {len(test_images)} images\n")

# Prepare data
X_train, y_train = prepare_data(train_images, train_labels, num_classes=62)
X_test, y_test   = prepare_data(test_images, test_labels, num_classes=62)

# Create and compile model
model_balanced = create_model()
model_balanced.summary()

optimizer = keras.optimizers.Adam(learning_rate=initial_lr)
model_balanced.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history_balanced = model_balanced.fit(
    X_train, y_train,
    batch_size=128,
    epochs=60,
    validation_split=0.1,
    callbacks=get_callbacks('emnist_balanced_model'),
    verbose=1
)

# Plot history
plot_history(history_balanced, 'Balanced Dataset')

# Evaluate
test_loss_bal, test_acc_bal = evaluate_model(model_balanced, X_test, y_test, 'BALANCED DATASET')

model_balanced.save('models/emnist_balanced.keras')

In [ ]:
print("TRAINING ON NOISY DATASET")

# Load data
train_images = load_idx_images('../data/noisy/emnist-byclass-train-images-idx3-ubyte')
train_labels = load_idx_labels('../data/noisy/emnist-byclass-train-labels-idx1-ubyte')
test_images = load_idx_images('../data/noisy/emnist-byclass-test-images-idx3-ubyte')
test_labels = load_idx_labels('../data/noisy/emnist-byclass-test-labels-idx1-ubyte')

print(f"Train: {len(train_images)} images")
print(f"Test: {len(test_images)} images\n")

# Prepare data
X_train, y_train = prepare_data(train_images, train_labels, num_classes=62)
X_test, y_test   = prepare_data(test_images, test_labels, num_classes=62)

# Create and compile model
model_noisy = create_model()
model_noisy.summary()

optimizer = keras.optimizers.Adam(learning_rate=initial_lr)
model_noisy.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history_noisy = model_noisy.fit(
    X_train, y_train,
    batch_size=128,
    epochs=60,
    validation_split=0.1,
    callbacks=get_callbacks('emnist_noisy_model'),
    verbose=1
)

# Plot history
plot_history(history_noisy, 'Noisy Dataset')

# Evaluate
test_loss_noisy, test_acc_noisy = evaluate_model(model_noisy, X_test, y_test, 'NOISY DATASET')

model_noisy.save('models/emnist_noisy.keras')

In [ ]:
print("TRAINING ON COMBINED DATASET")

# Load data
train_images = load_idx_images('../data/combined/emnist-byclass-train-images-idx3-ubyte')
train_labels = load_idx_labels('../data/combined/emnist-byclass-train-labels-idx1-ubyte')
test_images = load_idx_images('../data/combined/emnist-byclass-test-images-idx3-ubyte')
test_labels = load_idx_labels('../data/combined/emnist-byclass-test-labels-idx1-ubyte')

print(f"Train: {len(train_images)} images")
print(f"Test: {len(test_images)} images\n")

# Prepare data
X_train, y_train = prepare_data(train_images, train_labels, num_classes=62)
X_test, y_test   = prepare_data(test_images, test_labels, num_classes=62)

# Create and compile model
model_combined = create_model()
model_combined.summary()

optimizer = keras.optimizers.Adam(learning_rate=initial_lr)
model_combined.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history_combined = model_combined.fit(
    X_train, y_train,
    batch_size=128,
    epochs=60,
    validation_split=0.1,
    callbacks=get_callbacks('emnist_combined_model'),
    verbose=1
)

# Plot history
plot_history(history_combined, 'Combined Dataset')

# Evaluate
test_loss_comb, test_acc_comb = evaluate_model(model_combined, X_test, y_test, 'COMBINED DATASET')

model_combined.save('models/emnist_combined.keras')